In [2]:
from dataclasses import dataclass
from datetime import datetime
import queue

@dataclass
class Event:
    timestamp: datetime

@dataclass
class MarketEvent(Event):
    pass

@dataclass
class SentimentEvent(Event):
    ticker: str
    sentiment_score: float

@dataclass
class SignalEvent(Event):
    ticker: str
    direction: str
    strength: float = 1.0

@dataclass
class OrderEvent(Event):
    ticker: str
    order_type: str
    direction: str
    quantity: float
    price: float = 0.0

@dataclass
class FillEvent(Event):
    ticker: str
    direction: str
    quantity: float
    fill_price: float
    commission: float

In [3]:
import queue
import time
from datetime import datetime

class Backtest:
    """
    Encapsulates the settings and event loop for an event-driven backtest.
    """
    def __init__(
        self,
        data_handler,
        strategy,
        portfolio,
        execution_handler,
        verbose: bool = True
    ):
        self.data_handler = data_handler
        self.strategy = strategy
        self.portfolio = portfolio
        self.execution_handler = execution_handler
        self.verbose = verbose

        # Central Event Queue
        self.event_queue = queue.Queue()

        # Inject the shared event queue into all components
        self.data_handler.event_queue = self.event_queue
        self.strategy.event_queue = self.event_queue
        self.portfolio.event_queue = self.event_queue
        self.execution_handler.event_queue = self.event_queue

        self.events_processed = 0

    def run(self):
        """Executes the simulation loop."""
        if self.verbose:
            print("--- STARTING EVENT-DRIVEN BACKTEST ---")

        start_time = time.time()

        # Step through time bar-by-bar
        while self.data_handler.update_bars():
            # Process all cascade events for the current timestamp
            while not self.event_queue.empty():
                try:
                    event = self.event_queue.get_nowait()
                except queue.Empty:
                    break

                self.events_processed += 1

                if isinstance(event, MarketEvent):
                    # 1. Evaluate strategy rules
                    self.strategy.calculate_signals(event)
                    # 2. Update mark-to-market valuations
                    self.portfolio.update_timeindex(event)

                elif isinstance(event, SignalEvent):
                    # 3. Position sizing & risk management
                    self.portfolio.process_signal(event)

                elif isinstance(event, OrderEvent):
                    # 4. Route order to market/broker simulation
                    self.execution_handler.execute_order(event)

                elif isinstance(event, FillEvent):
                    # 5. Update cash, holdings, and transaction history
                    self.portfolio.process_fill(event)

        elapsed = time.time() - start_time
        if self.verbose:
            print(f"Backtest completed in {elapsed:.2f} seconds.")
            print(f"Total events processed: {self.events_processed}")
            print("---------------------------------------")

    def get_results(self):
        """Returns performance metrics and the equity curve DataFrame."""
        return self.portfolio.get_equity_curve()

In [4]:
def get_latest_bars(self, n=1):
    # Returns the current day's market data for all tickers.
    return self.current_data

In [5]:
import yfinance as yf
from sqlalchemy import create_engine

print("Downloading dummy data for SQLite Database...")
df = yf.download("AAPL", start="2023-01-01", end="2024-01-01", progress=False)

# Flatten columns, rename to match the DataHandler's expectations
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df.columns = [c.lower() for c in df.columns]
df['ticker'] = 'AAPL' # Add the ticker column

# Push to a local SQLite database in Colab
engine = create_engine('sqlite:///market_data.db')
df.to_sql('historical_prices', engine, index=False, if_exists='replace')

print("Success! Created market_data.db with table 'historical_prices'.")

Success! Created market_data.db with table 'historical_prices'.


In [6]:
from abc import ABC, abstractmethod
import pandas as pd
from sqlalchemy import create_engine

# Make sure MarketEvent is imported (assumes Cell 1 is run)
# from engine.event import MarketEvent

class DataHandler(ABC):
    @abstractmethod
    def get_latest_bars(self, ticker, n=1):
        raise NotImplementedError

    @abstractmethod
    def update_bars(self):
        raise NotImplementedError

class DatabaseDataHandler(DataHandler):
    def __init__(self, event_queue, db_path, table_name, tickers, start_date, end_date):
        self.event_queue = event_queue
        self.db_path = db_path
        self.table_name = table_name
        self.tickers = tickers
        self.start_date = pd.to_datetime(start_date)
        self.end_date = pd.to_datetime(end_date)

        # Connect to SQLite
        self.engine = create_engine(f'sqlite:///{self.db_path}')

        # State tracking
        self.all_data = None
        self.unique_dates = []
        self.historical_data = {ticker: [] for ticker in self.tickers} # FIX: Stores past bars

        self._load_data()
        self.data_generator = self._create_generator()

    def _load_data(self):
        placeholders = ', '.join('?' for _ in self.tickers)
        query = f"""
            SELECT * FROM {self.table_name}
            WHERE ticker IN ({placeholders})
            AND date BETWEEN ? AND ?
            ORDER BY date ASC
        """
        params = tuple(self.tickers + [self.start_date.strftime('%Y-%m-%d'), self.end_date.strftime('%Y-%m-%d')])

        self.all_data = pd.read_sql(
            query, self.engine, params=params,
            index_col='date', parse_dates=['date']
        )

        if self.all_data.empty:
            print(f"Warning: No data found for tickers {self.tickers}.")
        else:
            print(f"Data loaded successfully: {len(self.all_data)} rows.")

        self.unique_dates = self.all_data.index.unique()

    def _create_generator(self):
        for dt in self.unique_dates:
            yield dt, self.all_data.loc[dt]

    def get_latest_bars(self, ticker, n=1):
        """FIX: Returns the last N bars for a specific ticker as a list of dicts."""
        if ticker in self.historical_data:
            # Return the last N elements
            return self.historical_data[ticker][-n:]
        return []

    def update_bars(self):
        try:
            dt, data = next(self.data_generator)

            # Handle single ticker (Series) vs multiple tickers (DataFrame)
            if isinstance(data, pd.Series):
                data = data.to_frame().T

            data = data.reset_index() # Ensure 'date' is a column, not just the index

            # FIX: Append this new data to our rolling historical memory
            for _, row in data.iterrows():
                ticker = row['ticker']
                if ticker in self.historical_data:
                    self.historical_data[ticker].append(row.to_dict())

            self.event_queue.put(MarketEvent(dt))
            return True

        except StopIteration:
            return False

In [7]:
from abc import ABC, abstractmethod

# Make sure SignalEvent is imported (assumes Cell 1 is run)
# from engine.event import SignalEvent

class BaseStrategy(ABC):
    def __init__(self, data_handler, event_queue, params=None):
        self.data_handler = data_handler
        self.event_queue = event_queue
        self.params = params if params is not None else {}
        self.tickers = self.data_handler.tickers

        # State tracking to avoid spamming the Portfolio with duplicate signals
        self.bought = {ticker: False for ticker in self.tickers}

    @abstractmethod
    def calculate_signals(self, event):
        raise NotImplementedError("Should implement calculate_signals()")


class MomentumStrategy(BaseStrategy):
    """
    Cross-Sectional Momentum:
    Ranks stocks by their return over the 'lookback' period.
    Buys the top 2. Rebalances every 'rebalance_period' days.
    """
    def __init__(self, data_handler, event_queue, params):
        super().__init__(data_handler, event_queue, params)
        self.lookback = self.params.get('lookback_period', 90)
        self.rebalance_period = self.params.get('rebalance_period', 30)
        self.days_since_rebalance = 0

    def calculate_signals(self, event):
        date = event.timestamp.date()

        # 1. Advance the rebalance clock
        self.days_since_rebalance += 1

        # Only do heavy calculations on rebalance days
        if self.days_since_rebalance < self.rebalance_period:
            return

        # 2. Fetch exactly the data we need from the DataHandler
        momentum = {}
        for ticker in self.tickers:
            # Ask for lookback + 1 bars to get the start and end price
            bars = self.data_handler.get_latest_bars(ticker, self.lookback + 1)

            # Ensure we have enough historical data to calculate momentum
            if len(bars) == self.lookback + 1:
                current_price = bars[-1]['close']
                past_price = bars[0]['close']

                if past_price > 0:
                    returns = (current_price / past_price) - 1
                    momentum[ticker] = returns

        # If no tickers have met the lookback threshold yet, skip
        if len(momentum) < 2:
            return

        print(f"\n[{date}] DEBUG: Rebalancing... (Days since last: {self.days_since_rebalance})")
        self.days_since_rebalance = 0

        # 3. Rank the tickers
        ranked_tickers = sorted(momentum.items(), key=lambda x: x[1], reverse=True)
        # Select Top 2 performing assets
        long_list = [item[0] for item in ranked_tickers[:2]]

        print(f"[{date}] DEBUG: Momentum values: {{k: round(v, 4) for k, v in momentum.items()}}")
        print(f"[{date}] DEBUG: Top 2 momentum tickers: {long_list}")

        # 4. Generate Signals and update internal state
        for ticker in self.tickers:
            if ticker in long_list and not self.bought[ticker]:
                print(f"[{date}] DEBUG: --> Emitting LONG signal for {ticker}")
                self.event_queue.put(SignalEvent(event.timestamp, ticker, 'LONG'))
                self.bought[ticker] = True

            elif ticker not in long_list and self.bought[ticker]:
                print(f"[{date}] DEBUG: --> Emitting EXIT signal for {ticker}")
                self.event_queue.put(SignalEvent(event.timestamp, ticker, 'EXIT'))
                self.bought[ticker] = False

In [8]:
import numpy as np
from abc import ABC, abstractmethod

# Make sure SignalEvent is imported (assumes Cell 1 is run)
# from engine.event import SignalEvent

class BaseStrategy(ABC):
    def __init__(self, data_handler, event_queue, params=None):
        self.data_handler = data_handler
        self.event_queue = event_queue
        self.params = params if params is not None else {}
        self.tickers = self.data_handler.tickers

        # State tracking: None, 'LONG', or 'SHORT'
        self.position = {ticker: None for ticker in self.tickers}

    @abstractmethod
    def calculate_signals(self, event):
        raise NotImplementedError("Should implement calculate_signals()")


class MeanReversionStrategy(BaseStrategy):
    """
    Mean Reversion Strategy:
    Uses Z-scores to detect overbought/oversold conditions.
    """
    def __init__(self, data_handler, event_queue, params):
        super().__init__(data_handler, event_queue, params)
        self.lookback = self.params.get('lookback_period', 20)
        self.z_threshold = self.params.get('z_score_threshold', 2.0)
        self.rebalance_period = self.params.get('rebalance_period', 5)
        self.long_only = self.params.get('long_only', False) # Default to False to allow shorting
        self.days_since_rebalance = 0

    def calculate_signals(self, event):
        self.days_since_rebalance += 1
        if self.days_since_rebalance < self.rebalance_period:
            return  # Skip until next rebalance

        self.days_since_rebalance = 0

        for ticker in self.tickers:
            # 1. Fetch exactly the N bars we need from the DataHandler directly
            bars = self.data_handler.get_latest_bars(ticker, self.lookback)

            if len(bars) < self.lookback:
                continue  # Not enough data yet

            # 2. Extract closing prices into a fast NumPy array
            closes = np.array([bar['close'] for bar in bars])
            current_price = closes[-1]

            # 3. Calculate Z-Score mathematically without Pandas
            mean = np.mean(closes)
            std = np.std(closes, ddof=1) # ddof=1 for sample standard deviation

            if std == 0:
                continue

            z_score = (current_price - mean) / std

            # --- Signal Logic ---
            current_position = self.position[ticker]

            # Oversold -> LONG (Price is significantly below mean)
            if z_score < -self.z_threshold:
                if current_position != 'LONG':
                    # Dynamic sizing: cap strength at 3.0x
                    strength = float(min(abs(z_score) / self.z_threshold, 3.0))
                    self.event_queue.put(SignalEvent(event.timestamp, ticker, 'LONG', strength=strength))
                    self.position[ticker] = 'LONG'
                    print(f"[{event.timestamp.date()}] {ticker}: LONG (z={z_score:.2f}, strength={strength:.2f})")

            # Overbought -> SHORT or EXIT (Price is significantly above mean)
            elif z_score > self.z_threshold:
                if not self.long_only and current_position != 'SHORT':
                    # Open short position
                    strength = float(min(abs(z_score) / self.z_threshold, 3.0))
                    self.event_queue.put(SignalEvent(event.timestamp, ticker, 'SHORT', strength=strength))
                    self.position[ticker] = 'SHORT'
                    print(f"[{event.timestamp.date()}] {ticker}: SHORT (z={z_score:.2f}, strength={strength:.2f})")

                elif self.long_only and current_position == 'LONG':
                    # Exit long position (we don't short, just take profits)
                    self.event_queue.put(SignalEvent(event.timestamp, ticker, 'EXIT'))
                    self.position[ticker] = None
                    print(f"[{event.timestamp.date()}] {ticker}: EXIT (reverting from overbought)")

            # Mean Reversion -> EXIT (Price has returned to the mean)
            elif abs(z_score) < 0.5 and current_position is not None:
                self.event_queue.put(SignalEvent(event.timestamp, ticker, 'EXIT'))
                self.position[ticker] = None
                print(f"[{event.timestamp.date()}] {ticker}: EXIT (reverted to mean)")

In [9]:
import math
import pandas as pd
from datetime import datetime

# Make sure OrderEvent and FillEvent are imported (assumes Cell 1 is run)
# from engine.event import OrderEvent, FillEvent

# ==========================================
# 1. THE EXECUTION HANDLER (Broker Simulation)
# ==========================================
class SimulatedExecutionHandler:
    """
    Simulates a broker. Processes OrderEvents and generates FillEvents.
    Includes basic models for commissions and slippage.
    """
    def __init__(self, event_queue, commission_pct=0.001, slippage_pct=0.0005):
        self.event_queue = event_queue
        self.commission_pct = commission_pct
        self.slippage_pct = slippage_pct

    def execute_order(self, event):
        # In a real system, we'd query the LOB here. For now, assume the order
        # price is the current market price (with simulated slippage).
        # We'll pass the exact price in the OrderEvent for simplicity in simulation.

        # Apply Slippage
        if event.direction == 'BUY':
            fill_price = event.price * (1 + self.slippage_pct)
        else:
            fill_price = event.price * (1 - self.slippage_pct)

        # Calculate Commission
        commission = fill_price * event.quantity * self.commission_pct

        # Emit FillEvent
        fill_event = FillEvent(
            timestamp=event.timestamp,
            ticker=event.ticker,
            direction=event.direction,
            quantity=event.quantity,
            fill_price=fill_price,
            commission=commission
        )
        self.event_queue.put(fill_event)


# ==========================================
# 2. THE PORTFOLIO (Risk & Position Management)
# ==========================================
class Portfolio:
    """
    Tracks positions, cash, and total equity.
    Translates Strategy Signals into executable Orders based on risk rules.
    """
    def __init__(self, data_handler, event_queue, initial_capital=100000.0):
        self.data_handler = data_handler
        self.event_queue = event_queue
        self.initial_capital = float(initial_capital)

        self.current_positions = {ticker: 0.0 for ticker in self.data_handler.tickers}
        self.current_holdings = self._construct_initial_holdings()

        # Use a list of dicts for O(1) appending. Convert to DataFrame only at the end.
        self.all_holdings = []

    def _construct_initial_holdings(self):
        holdings = {'cash': self.initial_capital, 'total': self.initial_capital}
        for ticker in self.data_handler.tickers:
            holdings[ticker] = 0.0
        return holdings

    # -------- TIME INDEX UPDATES --------
    def update_timeindex(self, event):
        """Called every MarketEvent to update mark-to-market portfolio valuation."""
        date = event.timestamp
        market_value = 0.0

        for ticker in self.data_handler.tickers:
            qty = self.current_positions[ticker]
            if qty != 0:
                bar = self.data_handler.get_latest_bar(ticker)
                if bar:
                    close_price = bar['close']
                    # Value of the position (negative if shorting)
                    pos_value = qty * close_price
                    self.current_holdings[ticker] = pos_value
                    market_value += pos_value
            else:
                self.current_holdings[ticker] = 0.0

        self.current_holdings['total'] = self.current_holdings['cash'] + market_value

        # Save a snapshot of the current state
        snapshot = self.current_holdings.copy()
        snapshot['datetime'] = date
        self.all_holdings.append(snapshot)

    # -------- SIGNAL PROCESSING --------
    def process_signal(self, event):
        """Converts a SignalEvent into an OrderEvent with position sizing."""
        ticker = event.ticker
        direction = event.direction

        bar = self.data_handler.get_latest_bar(ticker)
        if not bar: return
        market_price = bar['close']

        # Base allocation: 10% of total equity, scaled by Strategy conviction strength
        target_value = self.current_holdings['total'] * 0.10 * event.strength
        quantity = math.floor(target_value / market_price)

        current_qty = self.current_positions[ticker]

        # 1. LONG SIGNAL (Buy to open)
        if direction == 'LONG' and current_qty == 0 and quantity > 0:
            order = OrderEvent(event.timestamp, ticker, 'MKT', 'BUY', quantity)
            order.price = market_price # Attach price for the simulated broker
            self.event_queue.put(order)

        # 2. SHORT SIGNAL (Sell to open)
        elif direction == 'SHORT' and current_qty == 0 and quantity > 0:
            order = OrderEvent(event.timestamp, ticker, 'MKT', 'SELL', quantity)
            order.price = market_price
            self.event_queue.put(order)

        # 3. EXIT SIGNAL (Close open positions)
        elif direction == 'EXIT' and current_qty != 0:
            exit_direction = 'SELL' if current_qty > 0 else 'BUY'
            order = OrderEvent(event.timestamp, ticker, 'MKT', exit_direction, abs(current_qty))
            order.price = market_price
            self.event_queue.put(order)

    # -------- FILL PROCESSING --------
    def process_fill(self, event):
        """Updates holdings and cash after receiving a FillEvent from the broker."""
        fill_cost = event.fill_price * event.quantity

        if event.direction == 'BUY':
            self.current_positions[event.ticker] += event.quantity
            self.current_holdings['cash'] -= (fill_cost + event.commission)
        elif event.direction == 'SELL':
            self.current_positions[event.ticker] -= event.quantity
            self.current_holdings['cash'] += (fill_cost - event.commission)

    # -------- EQUITY CURVE --------
    def get_equity_curve(self):
        """Builds the final performance DataFrame."""
        if not self.all_holdings:
            return pd.DataFrame()

        # Convert list of dicts to DataFrame in one fast operation
        equity_df = pd.DataFrame(self.all_holdings)
        equity_df.set_index('datetime', inplace=True)
        equity_df['returns'] = equity_df['total'].pct_change().fillna(0)
        equity_df['equity_curve'] = (1.0 + equity_df['returns']).cumprod()
        return equity_df


In [10]:
import numpy as np
import pandas as pd

def calculate_sharpe_ratio(returns, risk_free_rate=0.0):
    if returns.std() == 0: return 0.0
    excess_returns = returns - (risk_free_rate / 252)
    return np.sqrt(252) * (excess_returns.mean() / excess_returns.std())

def calculate_sortino_ratio(returns, risk_free_rate=0.0):
    excess_returns = returns - (risk_free_rate / 252)
    downside_returns = excess_returns[excess_returns < 0]
    if downside_returns.std() == 0: return 0.0
    return np.sqrt(252) * (excess_returns.mean() / downside_returns.std())

def calculate_max_drawdown(equity_curve_series):
    if len(equity_curve_series) < 2: return 0.0
    high_water_mark = equity_curve_series.cummax()
    drawdown = (equity_curve_series - high_water_mark) / high_water_mark
    return drawdown.min()

def generate_summary_stats(equity_df, risk_free_rate=0.0):
    equity_series = equity_df['total']
    returns = equity_df['returns']

    total_return = (equity_series.iloc[-1] / equity_series.iloc[0]) - 1

    return pd.DataFrame({
        "Metric": ["Total Return", "Max Drawdown", "Sharpe Ratio", "Sortino Ratio"],
        "Value": [
            f"{total_return * 100:.2f}%",
            f"{calculate_max_drawdown(equity_series) * 100:.2f}%",
            f"{calculate_sharpe_ratio(returns, risk_free_rate):.2f}",
            f"{calculate_sortino_ratio(returns, risk_free_rate):.2f}"
        ]
    }).set_index("Metric")

In [11]:
import pandas as pd
import yfinance as yf
from abc import ABC, abstractmethod

class DataHandler(ABC):
    @abstractmethod
    def get_latest_bar(self, ticker: str):
        pass
    @abstractmethod
    def get_latest_bars(self, ticker: str, N: int = 1):
        pass
    @abstractmethod
    def update_bars(self) -> bool:
        pass

class YFinanceDataHandler(DataHandler):
    """
    Streams historical data downloaded from Yahoo Finance
    bar-by-bar into the backtest engine.
    """
    def __init__(self, event_queue, tickers, start_date, end_date):
        self.event_queue = event_queue
        self.tickers = tickers
        self.start_date = start_date
        self.end_date = end_date

        self.symbol_data = {}
        self.latest_symbol_data = {ticker: [] for ticker in tickers}
        self.continue_backtest = True

        self._load_data()
        self._create_generators()

    def _load_data(self):
        for ticker in self.tickers:
            df = yf.download(ticker, start=self.start_date, end=self.end_date, progress=False)

            # Flatten columns for yfinance compatibility
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df = df.reset_index()
            df.columns = [c.lower() for c in df.columns]
            if 'date' in df.columns:
                df.rename(columns={'date': 'datetime'}, inplace=True)

            df.sort_values(by='datetime', inplace=True)
            self.symbol_data[ticker] = df

    def _create_generators(self):
        self.data_generators = {
            ticker: self.symbol_data[ticker].iterrows()
            for ticker in self.tickers
        }

    def get_latest_bar(self, ticker: str):
        try:
            return self.latest_symbol_data[ticker][-1]
        except IndexError:
            return {}

    def get_latest_bars(self, ticker: str, N: int = 1):
        try:
            return self.latest_symbol_data[ticker][-N:]
        except IndexError:
            return []

    def update_bars(self):
        for ticker in self.tickers:
            try:
                _, row = next(self.data_generators[ticker])
            except StopIteration:
                self.continue_backtest = False
                return False

            bar_dict = row.to_dict()
            self.latest_symbol_data[ticker].append(bar_dict)

            # Emit MarketEvent
            self.event_queue.put(MarketEvent(timestamp=bar_dict['datetime']))

        return True

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from datetime import datetime

# --- Backtest Setup ---
# Define parameters
tickers = ['AAPL', 'MSFT'] # Using tickers from kernel state
start_date = "2023-01-01"
end_date = "2024-01-01"
initial_capital = 100000.0

# Instantiate DataHandler
data_handler = YFinanceDataHandler(
    event_queue=None, # will be set by Backtest
    tickers=tickers,
    start_date=start_date,
    end_date=end_date
)

# Instantiate Strategy
strategy_params = {'lookback_period': 20, 'z_score_threshold': 1.5, 'rebalance_period': 10, 'long_only': False}
strategy = MeanReversionStrategy(
    data_handler=data_handler,
    event_queue=None, # will be set by Backtest
    params=strategy_params
)

# Instantiate Portfolio
portfolio = Portfolio(
    data_handler=data_handler,
    event_queue=None, # will be set by Backtest
    initial_capital=initial_capital
)

# Instantiate ExecutionHandler
execution_handler = SimulatedExecutionHandler(
    event_queue=None # will be set by Backtest
)

# Instantiate Backtest
backtest = Backtest(
    data_handler=data_handler,
    strategy=strategy,
    portfolio=portfolio,
    execution_handler=execution_handler,
    verbose=True
)

# Run the backtest
backtest.run()

# Get results
results_df = backtest.get_results()

def plot_interactive_dashboard(df):
    # Create a 2-row subplot (Top: Equity Curve, Bottom: Drawdown)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.1,
                        subplot_titles=('Portfolio Equity Curve', 'Drawdown (%)'),
                        row_heights=[0.7, 0.3])

    # Plot 1: Equity Curve
    fig.add_trace(
        go.Scatter(x=df.index, y=df['total'], mode='lines', name='Total Equity', line=dict(color='blue')),
        row=1, col=1
    )

    # Calculate Drawdown
    cum_max = df['total'].cummax()
    drawdown = (df['total'] / cum_max) - 1.0

    # Plot 2: Drawdown
    fig.add_trace(
        go.Scatter(x=df.index, y=drawdown * 100, mode='lines', name='Drawdown %',
                   line=dict(color='red'), fill='tozeroy'),
        row=2, col=1
    )

    # Formatting
    fig.update_layout(
        height=700,
        title_text="Event-Driven Backtest Results",
        template="plotly_white",
        showlegend=False
    )
    fig.update_yaxes(title_text="USD ($)", row=1, col=1)
    fig.update_yaxes(title_text="Percentage (%)", row=2, col=1)

    fig.show()

# Render the dashboard
plot_interactive_dashboard(results_df)

--- STARTING EVENT-DRIVEN BACKTEST ---
[2023-02-07] AAPL: SHORT (z=1.77, strength=1.18)
[2023-02-07] MSFT: SHORT (z=2.16, strength=1.44)
[2023-02-22] AAPL: EXIT (reverted to mean)
[2023-03-01] AAPL: LONG (z=-1.71, strength=1.14)
[2023-03-01] MSFT: LONG (z=-1.58, strength=1.06)
[2023-03-08] MSFT: EXIT (reverted to mean)
[2023-03-15] MSFT: SHORT (z=1.81, strength=1.21)
[2023-03-22] AAPL: SHORT (z=1.53, strength=1.02)
[2023-04-20] MSFT: EXIT (reverted to mean)
[2023-04-27] MSFT: SHORT (z=3.00, strength=2.00)
[2023-05-04] AAPL: EXIT (reverted to mean)
[2023-05-11] AAPL: SHORT (z=1.62, strength=1.08)
[2023-05-25] AAPL: EXIT (reverted to mean)
[2023-06-02] AAPL: SHORT (z=2.45, strength=1.63)
[2023-06-09] MSFT: EXIT (reverted to mean)
[2023-07-11] AAPL: EXIT (reverted to mean)
[2023-07-18] MSFT: SHORT (z=3.19, strength=2.13)
[2023-08-08] AAPL: LONG (z=-2.24, strength=1.49)
[2023-09-20] MSFT: LONG (z=-1.69, strength=1.13)
[2023-10-04] AAPL: EXIT (reverted to mean)
[2023-10-11] AAPL: SHORT (z=1

In [13]:
import queue
from dataclasses import dataclass, field
from datetime import datetime
import numpy as np

In [14]:
# ==========================================
# 1. UPGRADED PRIORITY EVENTS
# ==========================================
# We use order=True so the PriorityQueue can sort them chronologically
@dataclass(order=True)
class Event:
    timestamp: datetime
    # Tiebreaker ensures that if a MarketEvent and OrderEvent happen at the
    # exact same millisecond, they process in a logical order (Market first).
    priority: int

@dataclass(order=True)
class MarketEvent(Event):
    def __init__(self, timestamp):
        super().__init__(timestamp=timestamp, priority=1)

@dataclass(order=True)
class SentimentEvent(Event): # NEW: For Alternative Data
    ticker: str = field(compare=False)
    sentiment_score: float = field(compare=False)
    def __init__(self, timestamp, ticker, sentiment_score):
        super().__init__(timestamp=timestamp, priority=2)
        self.ticker = ticker
        self.sentiment_score = sentiment_score

@dataclass(order=True)
class SignalEvent(Event):
    ticker: str = field(compare=False)
    direction: str = field(compare=False)
    strength: float = field(compare=False, default=1.0)
    def __init__(self, timestamp, ticker, direction, strength=1.0):
        super().__init__(timestamp=timestamp, priority=3)
        self.ticker = ticker
        self.direction = direction
        self.strength = strength

@dataclass(order=True)
class OrderEvent(Event):
    ticker: str = field(compare=False)
    order_type: str = field(compare=False)
    direction: str = field(compare=False)
    quantity: float = field(compare=False)
    price: float = field(compare=False, default=0.0)
    def __init__(self, timestamp, ticker, order_type, direction, quantity, price=0.0):
        super().__init__(timestamp=timestamp, priority=4)
        self.ticker = ticker
        self.order_type = order_type
        self.direction = direction
        self.quantity = quantity
        self.price = price

@dataclass(order=True)
class FillEvent(Event):
    ticker: str = field(compare=False)
    direction: str = field(compare=False)
    quantity: float = field(compare=False)
    fill_price: float = field(compare=False)
    commission: float = field(compare=False)
    def __init__(self, timestamp, ticker, direction, quantity, fill_price, commission):
        super().__init__(timestamp=timestamp, priority=5)
        self.ticker = ticker
        self.direction = direction
        self.quantity = quantity
        self.fill_price = fill_price
        self.commission = commission

In [15]:
# ==========================================
# 2. THE RISK MANAGER
# ==========================================
class RiskManager:
    """
    Intercepts SignalEvents. Applies Inverse Volatility position sizing
    and checks global portfolio drawdown limits before emitting OrderEvents.
    """
    def __init__(self, data_handler, event_queue, max_drawdown=0.15):
        self.data_handler = data_handler
        self.event_queue = event_queue
        self.max_drawdown = max_drawdown

    def evaluate_signal(self, signal, portfolio):
        # 1. Global Risk Check: Stop trading if drawdown is too severe
        if self._is_drawdown_exceeded(portfolio):
            print(f"[{signal.timestamp.date()}] RISK MANAGER: Trade blocked. Max drawdown limit reached.")
            return

        # 2. Extract Data
        ticker = signal.ticker
        bar = self.data_handler.get_latest_bar(ticker)
        if not bar: return
        current_price = bar['close']

        # 3. Position Sizing: Inverse Volatility Sizing
        # We size trades inversely proportional to their 20-day volatility.
        # Highly volatile assets get smaller allocations to maintain Risk Parity.
        hist_bars = self.data_handler.get_latest_bars(ticker, 20)
        if len(hist_bars) < 20: return

        returns = pd.Series([b['close'] for b in hist_bars]).pct_change().dropna()
        volatility = returns.std() * np.sqrt(252) # Annualized Volatility

        if volatility == 0: return

        # Target risking exactly 1% of total equity based on the asset's volatility
        total_equity = portfolio.holdings['total']
        target_exposure = (total_equity * 0.01) / volatility

        # Scale by strategy conviction (strength)
        target_exposure *= signal.strength

        quantity = int(target_exposure / current_price)
        current_qty = portfolio.positions[ticker]

        # 4. Route Orders
        if signal.direction == 'LONG' and current_qty <= 0 and quantity > 0:
            order = OrderEvent(signal.timestamp, ticker, 'MKT', 'BUY', quantity, current_price)
            self.event_queue.put(order)

        elif signal.direction == 'SHORT' and current_qty >= 0 and quantity > 0:
            order = OrderEvent(signal.timestamp, ticker, 'MKT', 'SELL', quantity, current_price)
            self.event_queue.put(order)

        elif signal.direction == 'EXIT' and current_qty != 0:
            exit_dir = 'SELL' if current_qty > 0 else 'BUY'
            order = OrderEvent(signal.timestamp, ticker, 'MKT', exit_dir, abs(current_qty), current_price)
            self.event_queue.put(order)

    def _is_drawdown_exceeded(self, portfolio):
        total_equity = portfolio.holdings['total']
        # Simple high-water mark check
        if not hasattr(self, 'high_water_mark'):
            self.high_water_mark = total_equity

        if total_equity > self.high_water_mark:
            self.high_water_mark = total_equity

        drawdown = (self.high_water_mark - total_equity) / self.high_water_mark
        return drawdown >= self.max_drawdown

In [16]:
import pandas as pd

class LimitOrderBook:
    """
    Simulates a Level 2 exchange matching engine.
    Maintains synthetic order queues and executes trades based on Price-Time Priority,
    accounting for market impact and liquidity depletion.
    """
    def __init__(self, event_queue):
        self.event_queue = event_queue

        # State tracking for liquidity
        self.asks = []       # Sorted list of ask prices (ascending)
        self.bids = []       # Sorted list of bid prices (descending)
        self.ask_book = {}   # price -> available quantity
        self.bid_book = {}   # price -> available quantity

    def _populate_synthetic_liquidity(self, current_price):
        """
        Generates a synthetic order book around the current market price.
        Assumes there are 500 shares available at every $0.05 price increment.
        """
        # Create Ask levels (sellers offering above current price)
        self.asks = [round(current_price + (i * 0.05), 2) for i in range(1, 15)]
        self.ask_book = {price: 500 for price in self.asks}

        # Create Bid levels (buyers bidding below current price)
        self.bids = [round(current_price - (i * 0.05), 2) for i in range(1, 15)]
        self.bid_book = {price: 500 for price in self.bids}

    def execute_order(self, event):
        """Processes OrderEvents by matching them against the order book."""

        # 1. Build the order book for this specific timestamp
        self._populate_synthetic_liquidity(event.price)

        remaining_qty = event.quantity
        fills = []

        # 2. MATCH MARKET BUY (Walks UP the Ask book)
        if event.direction == 'BUY':
            for ask_price in sorted(self.asks):
                if remaining_qty <= 0:
                    break

                available_qty = self.ask_book[ask_price]
                fill_qty = min(remaining_qty, available_qty)

                if fill_qty > 0:
                    fills.append((ask_price, fill_qty))
                    self.ask_book[ask_price] -= fill_qty
                    remaining_qty -= fill_qty

        # 3. MATCH MARKET SELL (Walks DOWN the Bid book)
        elif event.direction == 'SELL':
            # Reverse sort so we hit the highest bids first
            for bid_price in sorted(self.bids, reverse=True):
                if remaining_qty <= 0:
                    break

                available_qty = self.bid_book[bid_price]
                fill_qty = min(remaining_qty, available_qty)

                if fill_qty > 0:
                    fills.append((bid_price, fill_qty))
                    self.bid_book[bid_price] -= fill_qty
                    remaining_qty -= fill_qty

        # 4. Calculate Final Execution via Volume-Weighted Average Price (VWAP)
        if fills:
            total_filled = sum(qty for _, qty in fills)
            # VWAP Formula
            vwap = sum(price * qty for price, qty in fills) / total_filled

            # Institutional commission estimate ($0.005 per share)
            commission = total_filled * 0.005

            print(f"[{event.timestamp.date()}] LOB MATCH: {event.direction} {total_filled} {event.ticker} @ VWAP ${vwap:.2f} (Crossed {len(fills)} liquidity levels)")

            # Emit the ground-truth FillEvent back to the Portfolio
            self.event_queue.put(FillEvent(
                timestamp=event.timestamp,
                ticker=event.ticker,
                direction=event.direction,
                quantity=total_filled,
                fill_price=vwap,
                commission=commission
            ))
        else:
            print(f"[{event.timestamp.date()}] LOB REJECT: Insufficient liquidity for {event.ticker}")

In [17]:
import pandas as pd
import numpy as np

class RiskManager:
    """
    Intercepts SignalEvents. Applies Inverse Volatility position sizing
    and checks global portfolio drawdown limits before emitting OrderEvents.
    """
    def __init__(self, data_handler, event_queue, max_drawdown=0.15):
        self.data_handler = data_handler
        self.event_queue = event_queue
        self.max_drawdown = max_drawdown

    def evaluate_signal(self, signal, portfolio):
        # 1. Global Risk Check: Stop trading if drawdown is too severe
        if self._is_drawdown_exceeded(portfolio):
            print(f"[{signal.timestamp.date()}] RISK MANAGER: Trade blocked. Max drawdown limit reached.")
            return

        # 2. Extract Data
        ticker = signal.ticker
        bar = self.data_handler.get_latest_bar(ticker)
        if not bar: return
        current_price = bar['close']

        # 3. Position Sizing: Inverse Volatility Sizing
        hist_bars = self.data_handler.get_latest_bars(ticker, 20)
        if len(hist_bars) < 20: return

        returns = pd.Series([b['close'] for b in hist_bars]).pct_change().dropna()
        volatility = returns.std() * np.sqrt(252)

        if volatility == 0: return

        # FIX: Use 'current_holdings' as defined in your Portfolio class
        total_equity = portfolio.current_holdings['total']
        target_exposure = (total_equity * 0.01) / volatility

        target_exposure *= signal.strength
        quantity = int(target_exposure / current_price)

        # FIX: Use 'current_positions' as defined in your Portfolio class
        current_qty = portfolio.current_positions[ticker]

        # 4. Route Orders
        if signal.direction == 'LONG' and current_qty <= 0 and quantity > 0:
            order = OrderEvent(signal.timestamp, ticker, 'MKT', 'BUY', quantity, current_price)
            self.event_queue.put(order)

        elif signal.direction == 'SHORT' and current_qty >= 0 and quantity > 0:
            order = OrderEvent(signal.timestamp, ticker, 'MKT', 'SELL', quantity, current_price)
            self.event_queue.put(order)

        elif signal.direction == 'EXIT' and current_qty != 0:
            exit_dir = 'SELL' if current_qty > 0 else 'BUY'
            order = OrderEvent(signal.timestamp, ticker, 'MKT', exit_dir, abs(current_qty), current_price)
            self.event_queue.put(order)

    def _is_drawdown_exceeded(self, portfolio):
        # FIX: Use 'current_holdings'
        total_equity = portfolio.current_holdings['total']
        if not hasattr(self, 'high_water_mark'):
            self.high_water_mark = total_equity

        if total_equity > self.high_water_mark:
            self.high_water_mark = total_equity

        drawdown = (self.high_water_mark - total_equity) / self.high_water_mark
        return drawdown >= self.max_drawdown

In [18]:
# import queue

# # ==========================================
# # THE UPGRADED CORE ENGINE
# # ==========================================
# class Backtest:
#     """
#     The central event loop that orchestrates the PriorityQueue
#     and strictly enforces chronological event processing.
#     """
#     def __init__(self, data_handler, strategy, risk_manager, portfolio, execution):
#         self.dh = data_handler
#         self.strategy = strategy
#         self.risk_manager = risk_manager
#         self.portfolio = portfolio
#         self.execution = execution
#         self.eq = data_handler.event_queue
#         self.events_processed = 0

#     def run(self):
#         print("--- STARTING INSTITUTIONAL EVENT-DRIVEN ENGINE ---")
#         while self.dh.update_bars():
#             # Process all cascaded events for this timestamp in priority order
#             while not self.eq.empty():
#                 try:
#                     event = self.eq.get_nowait()
#                 except queue.Empty:
#                     break

#                 self.events_processed += 1

#                 # Routing Logic
#                 if isinstance(event, MarketEvent):
#                     self.strategy.calculate_signals(event)
#                     self.portfolio.update_timeindex(event)

#                 elif isinstance(event, SignalEvent):
#                     # NEW: Signals must pass through the Risk Manager
#                     self.risk_manager.evaluate_signal(event, self.portfolio)

#                 elif isinstance(event, OrderEvent):
#                     # NEW: Orders are matched against the LOB Simulator
#                     self.execution.execute_order(event)

#                 elif isinstance(event, FillEvent):
#                     self.portfolio.process_fill(event)

#         print(f"--- BACKTEST COMPLETE ({self.events_processed} events processed) ---")

# # ==========================================
# # INITIALIZATION & EXECUTION
# # ==========================================
# # 1. Initialize the new PriorityQueue
# events = queue.PriorityQueue()

# print("Downloading Yahoo Finance Data...")
# # 2. Setup Data Feed
# tickers = ['AAPL', 'MSFT']
# data_handler = YFinanceDataHandler(events, tickers, '2023-01-01', '2024-01-01')

# # 3. Setup Strategy
# strategy_params = {'lookback_period': 20, 'z_score_threshold': 2.0}
# strategy = MeanReversionStrategy(data_handler, events, strategy_params)

# # 4. Setup Risk, Portfolio, and Execution (LOB)
# risk_manager = RiskManager(data_handler, events, max_drawdown=0.15)
# portfolio = Portfolio(data_handler, events)
# limit_order_book = LimitOrderBook(events)

# # 5. Build and Run the Engine (using the newly defined Backtest class above)
# engine = Backtest(data_handler, strategy, risk_manager, portfolio, limit_order_book)
# engine.run()

# # 6. Extract Results and Plot
# results_df = portfolio.get_equity_curve()
# if not results_df.empty:
#     plot_interactive_dashboard(results_df)

In [19]:
# ==========================================
# UPGRADED TUPLE-BASED PRIORITY QUEUE HELPER
# ==========================================
class SafePriorityQueue:
    """
    Wraps queue.PriorityQueue to ensure events are pushed as
    (priority, timestamp, event_object) tuples.
    """
    def __init__(self):
        self._q = queue.PriorityQueue()
        self._counter = 0 # Tie-breaker for identical timestamps

    def put(self, event):
        # Assign priorities based on event type
        if isinstance(event, MarketEvent):
            priority = 1
        elif isinstance(event, SentimentEvent):
            priority = 2
        elif isinstance(event, SignalEvent):
            priority = 3
        elif isinstance(event, OrderEvent):
            priority = 4
        elif isinstance(event, FillEvent):
            priority = 5
        else:
            priority = 100

        # Tuple structure: (priority, timestamp, counter, event)
        # The counter ensures stable sorting if timestamps match down to the tick.
        self._q.put((priority, event.timestamp, self._counter, event))
        self._counter += 1

    def get_nowait(self):
        # Unpack the tuple and return just the event object
        _, _, _, event = self._q.get_nowait()
        return event

    def empty(self):
        return self._q.empty()


# ==========================================
# THE UPGRADED CORE ENGINE
# ==========================================
class Backtest:
    def __init__(self, data_handler, strategy, risk_manager, portfolio, execution):
        self.dh = data_handler
        self.strategy = strategy
        self.risk_manager = risk_manager
        self.portfolio = portfolio
        self.execution = execution
        self.eq = data_handler.event_queue
        self.events_processed = 0

    def run(self):
        print("--- STARTING INSTITUTIONAL EVENT-DRIVEN ENGINE ---")
        while self.dh.update_bars():
            while not self.eq.empty():
                try:
                    event = self.eq.get_nowait()
                except queue.Empty:
                    break

                self.events_processed += 1

                if isinstance(event, MarketEvent):
                    self.strategy.calculate_signals(event)
                    self.portfolio.update_timeindex(event)
                elif isinstance(event, SignalEvent):
                    self.risk_manager.evaluate_signal(event, self.portfolio)
                elif isinstance(event, OrderEvent):
                    self.execution.execute_order(event)
                elif isinstance(event, FillEvent):
                    self.portfolio.process_fill(event)

        print(f"--- BACKTEST COMPLETE ({self.events_processed} events processed) ---")


# ==========================================
# INITIALIZATION & EXECUTION
# ==========================================
events = SafePriorityQueue()

print("Downloading Yahoo Finance Data...")
tickers = ['AAPL', 'MSFT']
data_handler = YFinanceDataHandler(events, tickers, '2023-01-01', '2024-01-01')

strategy_params = {'lookback_period': 20, 'z_score_threshold': 2.0}
strategy = MeanReversionStrategy(data_handler, events, strategy_params)

risk_manager = RiskManager(data_handler, events, max_drawdown=0.15)
portfolio = Portfolio(data_handler, events)
limit_order_book = LimitOrderBook(events)

engine = Backtest(data_handler, strategy, risk_manager, portfolio, limit_order_book)
engine.run()

results_df = portfolio.get_equity_curve()
if not results_df.empty:
    plot_interactive_dashboard(results_df)

--- STARTING INSTITUTIONAL EVENT-DRIVEN ENGINE ---
[2023-02-03] AAPL: SHORT (z=2.17, strength=1.08)
[2023-02-03] LOB MATCH: SELL 34 AAPL @ VWAP $151.74 (Crossed 1 liquidity levels)
[2023-02-07] MSFT: SHORT (z=2.16, strength=1.08)
[2023-02-07] LOB MATCH: SELL 11 MSFT @ VWAP $260.10 (Crossed 1 liquidity levels)
[2023-02-17] MSFT: EXIT (reverted to mean)
[2023-02-17] LOB MATCH: BUY 11.0 MSFT @ VWAP $251.60 (Crossed 1 liquidity levels)
[2023-02-22] AAPL: EXIT (reverted to mean)
[2023-02-22] LOB MATCH: BUY 34.0 AAPL @ VWAP $146.57 (Crossed 1 liquidity levels)
[2023-03-20] AAPL: SHORT (z=2.01, strength=1.00)
[2023-03-20] LOB MATCH: SELL 28 AAPL @ VWAP $154.83 (Crossed 1 liquidity levels)
[2023-04-11] AAPL: EXIT (reverted to mean)
[2023-04-11] LOB MATCH: BUY 28.0 AAPL @ VWAP $158.27 (Crossed 1 liquidity levels)
[2023-04-25] MSFT: LONG (z=-2.22, strength=1.11)
[2023-04-25] LOB MATCH: BUY 18 MSFT @ VWAP $268.52 (Crossed 1 liquidity levels)
[2023-04-27] MSFT: SHORT (z=3.00, strength=1.50)
[2023-

In [20]:
import os
import sqlite3

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

DB_PATH = r'c:\Users\Chayan\Desktop\backcrawl\market_data.db'

def plot_interactive_dashboard(df):
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=('Portfolio Equity Curve', 'Drawdown (%)'),
        row_heights=[0.7, 0.3]
    )

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['total'],
            mode='lines',
            name='Total Equity',
            line=dict(color='blue')
        ),
        row=1,
        col=1
    )

    cum_max = df['total'].cummax()
    drawdown = (df['total'] / cum_max) - 1.0

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=drawdown * 100,
            mode='lines',
            name='Drawdown %',
            line=dict(color='red'),
            fill='tozeroy'
        ),
        row=2,
        col=1
    )

    fig.update_layout(
        height=700,
        title_text='Event-Driven Backtest Results',
        template='plotly_white',
        showlegend=False
    )
    fig.update_yaxes(title_text='USD ($)', row=1, col=1)
    fig.update_yaxes(title_text='Percentage (%)', row=2, col=1)
    fig.show()

results = globals().get('results_df')

if results is not None and not results.empty:
    if 'generate_summary_stats' in globals():
        display(generate_summary_stats(results))
    plot_interactive_dashboard(results)
elif os.path.exists(DB_PATH):
    try:
        with sqlite3.connect(DB_PATH) as connection:
            market_data = pd.read_sql_query(
                'SELECT * FROM historical_prices',
                connection,
                parse_dates=['date']
            )

        market_data = market_data.sort_values(['ticker', 'date'])

        fig = make_subplots(
            rows=1,
            cols=1,
            subplot_titles=('Market Data Overview',)
        )

        for ticker, group in market_data.groupby('ticker'):
            fig.add_trace(
                go.Scatter(
                    x=group['date'],
                    y=group['close'],
                    mode='lines',
                    name=ticker
                ),
                row=1,
                col=1
            )

        fig.update_layout(
            height=600,
            title_text='Market Data Overview',
            template='plotly_white'
        )
        fig.update_yaxes(title_text='Close Price ($)', row=1, col=1)
        fig.show()
    except Exception as exc:
        print(f'Unable to render a visualization from disk: {exc}')
else:
    print('No results_df found and market_data.db is missing.')

,Value
Metric,
Total Return,0.72%
Max Drawdown,-0.82%
Sharpe Ratio,0.56
Sortino Ratio,0.45
